# PyTorch on a Colab TPU

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/06-PyTorch-on-TPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

T4 GPUs on Colab are not always available, especially on the free tier. Colab also offers **TPUs** (Google's Tensor Processing Units), and since October 2026 PyTorch runs on them **natively** through [TorchTPU](https://github.com/google-pytorch/torch_tpu): a TPU is just another device, `"tpu"`, like `"cuda"`.

**How to get one**: `Runtime → Change runtime type → v5e-1 TPU`. The free tier offers a single TPU v5e chip (`v5e-1`); paid plans can also choose `v6e-1`. The TPU runtime comes with `torch` (CPU build) and `torch_tpu` preinstalled, so there is nothing to install.

This notebook **needs a TPU runtime**: it runs a few examples from earlier notebooks on the TPU and compares them with the T4.

| | T4 GPU | TPU v5e (one chip) |
|---|---|---|
| Memory | 15 GB | 16 GB |
| Fast low precision | fp16 (Tensor Cores) | **bf16** (matrix units, "MXU") |
| Mixed precision | `autocast(float16)` + `GradScaler` | `autocast(bfloat16)`, no `GradScaler` needed |
| How code runs | CUDA kernels, op by op | ops are compiled by XLA; `torch.compile(backend="tpu")` compiles whole graphs |
| Host CPU on Colab | 2 cores | many more (24 when this notebook was written) |

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*non-leaf Tensor.*")   # harmless warning printed by TorchTPU 0.2
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
print(torch.__version__)

## What is different about a TPU, from a PyTorch user's point of view?

* **It is built for big matrix multiplications.** Each TPU v5e chip has matrix units (MXUs) that multiply 128x128 blocks in bf16. Large dense layers, convolutions and attention run very fast; dimensions that are multiples of 128 (or at least 8) use the hardware best, odd sizes get padded.
* **Code is compiled, not launched op by op.** On a GPU each PyTorch operation calls a ready-made CUDA kernel. On a TPU, operations go through the **XLA** compiler, which compiles them for the exact tensor shapes it sees, and `torch.compile(backend="tpu")` hands it whole graphs to optimize at once. The first call of anything is slow (compiling), and **changing shapes means compiling again**.
* **bf16 is the native low-precision type**, not fp16. bf16 has the same exponent range as fp32, so mixed precision needs no `GradScaler`.
* **The PyTorch backend is young.** TorchTPU appeared in 2026. Most of PyTorch works, but some operators are missing and some behave differently; we will meet both below. Libraries written for CUDA do not work at all.

## Device-agnostic code with `torch.accelerator`

Instead of hard-coding `"cuda"`, ask PyTorch which accelerator is present. The same code then runs on a T4 (`cuda`), a TPU (`tpu`), an Apple GPU (`mps`) or the CPU.

`torch.accelerator.synchronize()` waits until queued work has finished; use it before measuring time, just like `torch.cuda.synchronize()` in 01.

In [ ]:
acc = torch.accelerator.current_accelerator()
device = acc.type if acc is not None else "cpu"
print("device:", device)
if device == "tpu":
    print(torch.tpu.get_device_name())

def sync():
    if acc is not None:
        torch.accelerator.synchronize()

# the fast low-precision type: bf16 on TPUs, fp16 on a T4
AMP_DTYPE = torch.bfloat16 if device == "tpu" else torch.float16
x = torch.ones(3, device=device)
x * 2

## Example from 01: matrix multiplication speed

In [ ]:
def benchmark(dtype, n=4096, repeat=20):
    a = torch.randn(n, n, device=device, dtype=dtype)
    b = torch.randn(n, n, device=device, dtype=dtype)
    (a @ b).sum().item()   # warm up (the first call also compiles the operation)
    sync()
    t0 = time.perf_counter()
    for _ in range(repeat):
        c = a @ b
    sync()
    dt = (time.perf_counter() - t0) / repeat
    print(f"{device} {dtype}: {dt*1000:.2f} ms, {2*n**3/dt/1e12:.1f} TFLOPS")

for dtype in [torch.float32, torch.bfloat16]:
    benchmark(dtype)

On a T4, fp16 reaches about 30 TFLOPS (see 01). A single TPU v5e chip reaches roughly 5x that in bf16.
By default the TPU also computes float32 matrix multiplications with bf16 passes internally, which is why fp32 is fast too; this slightly reduces precision (the same idea as TF32 on newer NVIDIA GPUs).

## Example from 02: a CNN on MNIST

The code is the same as before; only `device` changed. The whole dataset sits on the TPU.

In [ ]:
from torchvision.datasets import MNIST
train_ds, test_ds = MNIST("data", train=True, download=True), MNIST("data", train=False, download=True)
train_X = (train_ds.data.float() / 255).reshape(-1, 784).to(device)
train_y = train_ds.targets.to(device)
test_X = (test_ds.data.float() / 255).reshape(-1, 784).to(device)
test_y = test_ds.targets.to(device)

def cnn_model():
    return nn.Sequential(
        nn.Unflatten(1, (1, 28, 28)),
        nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(64 * 7 * 7, 1024), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(1024, 10),
    )

@torch.no_grad()
def accuracy(model, X, y, batch_size=1000):
    model.eval()
    correct = sum((model(X[i:i + batch_size]).argmax(1) == y[i:i + batch_size]).sum() for i in range(0, len(X), batch_size))
    return correct.item() / len(X)

model = cnn_model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 256
for epoch in range(3):
    model.train()
    t0 = time.perf_counter()
    perm = torch.randperm(len(train_X), device=device)
    for i in range(0, len(train_X) - batch_size + 1, batch_size):   # drop the last partial batch (see below)
        idx = perm[i:i + batch_size]
        loss = F.cross_entropy(model(train_X[idx]), train_y[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    sync()
    print(f"epoch {epoch}  loss={loss.item():.4f}  test acc={accuracy(model, test_X, test_y):.4f}  {time.perf_counter()-t0:.1f}s")

Note: the first epoch is slower, because each new operation (and each new **tensor shape**) is compiled the first time it is used. Keeping shapes fixed avoids recompiling, which is why the loop drops the last, smaller batch.

## Example from 03: ResNet9 on CIFAR10, eager vs. `torch.compile`

First, something you will meet on a new platform: **not every operator is implemented yet**. The ResNet9 in 03 ends with `nn.AdaptiveMaxPool2d(1)`, whose backward pass TorchTPU 0.2 does not support:

In [ ]:
try:
    layer = nn.AdaptiveMaxPool2d(1).to(device)
    x = torch.randn(2, 8, 4, 4, device=device, requires_grad=True)
    layer(x).sum().backward()
    print("AdaptiveMaxPool2d backward works now")
except RuntimeError as e:
    print("RuntimeError:", str(e)[:150])

The input to that layer is always 4x4, so `nn.MaxPool2d(4)` computes exactly the same thing and is supported. This kind of small workaround is typical when moving code to a new accelerator.

A second, nastier surprise: in TorchTPU 0.2, **BatchNorm does not update its running statistics** during training (`running_mean` stays 0 and `running_var` stays 1). Training itself is fine, because in `train()` mode BatchNorm normalizes with the statistics of the current batch; but in `eval()` mode the model then normalizes with the wrong numbers and test accuracy collapses. There is no error message, only a bad result:

In [ ]:
bn = nn.BatchNorm2d(4).to(device).train()
for _ in range(20):
    bn(torch.randn(64, 4, 8, 8, device=device) * 3 + 5)     # data with mean 5
print("running_mean after 20 training batches:", bn.running_mean.cpu().tolist(), "(should be close to 5)")

The workaround: after training, **recompute** the statistics ourselves. Forward hooks collect the mean and variance of each BatchNorm layer's input over some training batches, and we write them into `running_mean` and `running_var`. (Recomputing BatchNorm statistics is also a standard trick after weight averaging, see `torch.optim.swa_utils.update_bn`.)

In [ ]:
@torch.no_grad()
def recalibrate_batchnorm(model, X, batch_size=500, n_batches=20):
    """Recompute BatchNorm running statistics from n_batches batches of X (works around TorchTPU 0.2)"""
    bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
    sums = {m: [0.0, 0.0] for m in bns}
    def hook(m, inputs):
        x = inputs[0].float()
        dims = [0] + list(range(2, x.dim()))          # all dimensions except the channels
        sums[m][0] = sums[m][0] + x.mean(dims)
        sums[m][1] = sums[m][1] + (x * x).mean(dims)
    handles = [m.register_forward_pre_hook(hook) for m in bns]
    model.train()
    for i in range(n_batches):
        model(X[i * batch_size:(i + 1) * batch_size])
    for h in handles:
        h.remove()
    for m in bns:
        mean = sums[m][0] / n_batches
        m.running_mean.copy_(mean)
        m.running_var.copy_(sums[m][1] / n_batches - mean ** 2)
    model.eval()

In [ ]:
from datasets import load_dataset
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
def load_cifar10(split):
    ds = load_dataset("uoft-cs/cifar10", split=split)
    X = torch.tensor(np.stack([np.array(im) for im in ds["img"]])).permute(0, 3, 1, 2).float() / 255
    return ((X - mean) / std).to(device), torch.tensor(ds["label"]).to(device)
cifar_train_X, cifar_train_y = load_cifar10("train")
cifar_test_X, cifar_test_y = load_cifar10("test")

def augment(X, pad=4):
    """random crop + horizontal flip on a whole batch (the gpu_augment from 03, unchanged)"""
    N, C, H, W = X.shape
    Xp = F.pad(X, (pad,) * 4, mode="reflect")
    dx, dy = torch.randint(0, 2 * pad + 1, (2, N), device=X.device)
    rows = torch.arange(H, device=X.device)[None, :] + dy[:, None]
    cols = torch.arange(W, device=X.device)[None, :] + dx[:, None]
    X = Xp[torch.arange(N, device=X.device)[:, None, None, None], torch.arange(C, device=X.device)[None, :, None, None],
           rows[:, None, :, None], cols[:, None, None, :]]
    flip = torch.rand(N, device=X.device) < 0.5
    return torch.where(flip[:, None, None, None], X.flip(3), X)

def conv_bn(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.GELU())

class Residual(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(conv_bn(c, c), conv_bn(c, c))
    def forward(self, x):
        return x + self.block(x)

def resnet9(num_classes=10, w=64):
    return nn.Sequential(
        conv_bn(3, w),
        conv_bn(w, 2 * w), nn.MaxPool2d(2), Residual(2 * w),
        conv_bn(2 * w, 4 * w), nn.MaxPool2d(2),
        conv_bn(4 * w, 8 * w), nn.MaxPool2d(2), Residual(8 * w),
        nn.MaxPool2d(4), nn.Flatten(),          # instead of nn.AdaptiveMaxPool2d(1)
        nn.Linear(8 * w, num_classes),
    )

The training loop is the one from 03, with two changes: **bf16** autocast (so no `GradScaler`), and every batch has the **same shape** (`drop_last`).
`compile=True` wraps the model in `torch.compile(model, backend="tpu")`: the whole forward and backward pass becomes one optimized XLA program. The first steps are slow while it compiles; after that it is faster than eager mode.

In [ ]:
@torch.no_grad()
def evaluate(model, batch_size=1000):
    model.eval()
    correct = 0
    for i in range(0, len(cifar_test_X), batch_size):
        with torch.autocast(device, dtype=AMP_DTYPE):
            correct += (model(cifar_test_X[i:i + batch_size]).argmax(1) == cifar_test_y[i:i + batch_size]).sum()
    return correct.item() / len(cifar_test_X)

def fit(model, epochs=10, batch_size=512, lr=2e-3, compile=False):
    model = model.to(device)
    train_model = torch.compile(model, backend=device) if compile and device == "tpu" else model
    steps_per_epoch = len(cifar_train_X) // batch_size
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * steps_per_epoch)
    for epoch in range(epochs):
        model.train()
        t0 = time.perf_counter()
        perm = torch.randperm(len(cifar_train_X), device=device)
        for s in range(steps_per_epoch):
            idx = perm[s * batch_size:(s + 1) * batch_size]
            with torch.autocast(device, dtype=AMP_DTYPE):
                loss = F.cross_entropy(train_model(augment(cifar_train_X[idx])), cifar_train_y[idx], label_smoothing=0.1)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            scheduler.step()
        sync()
        dt = time.perf_counter() - t0
        if device == "tpu":
            recalibrate_batchnorm(model, cifar_train_X)     # the BatchNorm workaround from above
        print(f"epoch {epoch:2d}  loss={loss.item():.3f}  test acc={evaluate(model):.4f}  {dt:.1f}s "
              f"({steps_per_epoch * batch_size / dt:,.0f} img/s)")
    return model

print("eager mode:")
fit(resnet9(), epochs=2)
print("torch.compile:")
model = fit(resnet9(), epochs=10, compile=True)

On the T4 in 03, one epoch takes about 9 seconds (about 5,500 img/s). Compare with the numbers above: here the first epoch includes the compilation, and the following ones show the real speed.

## Example from 05: training the character-level GPT

The same model as in 05 (attention with `F.scaled_dot_product_attention`, RMSNorm, weight tying), trained with bf16 and `torch.compile`.

In [ ]:
from urllib.request import urlopen
text = urlopen("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt").read().decode("utf8")
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n].to(device), data[n:].to(device)
block_size, batch_size = 256, 64

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,), device=device)
    offsets = torch.arange(block_size, device=device)
    return d[ix[:, None] + offsets], d[ix[:, None] + offsets + 1]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.norm1, self.norm2 = nn.RMSNorm(n_embd), nn.RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd, bias=False), nn.GELU(),
                                 nn.Linear(4 * n_embd, n_embd, bias=False), nn.Dropout(dropout))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer=6, n_head=6, n_embd=384, dropout=0.2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.norm = nn.RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.apply(lambda m: nn.init.normal_(m.weight, std=0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)
    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        return self.lm_head(self.norm(self.blocks(x)))

gpt = GPT(len(chars), block_size).to(device)
train_gpt = torch.compile(gpt, backend=device) if device == "tpu" else gpt
max_steps, lr = 1500, 1e-3
optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=max_steps, pct_start=0.05)
t0 = time.perf_counter()
for step in range(1, max_steps + 1):
    gpt.train()
    xb, yb = get_batch("train")
    with torch.autocast(device, dtype=AMP_DTYPE):
        logits = train_gpt(xb)
    loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), yb.view(-1))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    if step % 250 == 0:
        print(f"step {step:5d}  train loss {loss.item():.3f}  {time.perf_counter()-t0:.0f}s")

On the T4, 05 needs about 3 minutes for these 1500 steps.

Generating text adds one token at a time, so the sequence length changes at every step. With eager mode (the uncompiled `gpt`) that is fine:

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.8):
    model.eval()
    for _ in range(max_new_tokens):
        with torch.autocast(device, dtype=AMP_DTYPE):
            logits = model(idx[:, -block_size:])[:, -1, :].float()
        idx = torch.cat([idx, torch.multinomial(F.softmax(logits / temperature, dim=-1), 1)], dim=1)
    return idx

start = torch.tensor([encode("ROMEO:\n")], device=device)
print(decode(generate(gpt, start, 300)[0].tolist()))

## Example from 06-transformers: an LLM on the TPU

🤗 Transformers supports the TPU directly: `device_map="tpu"`. Load the model in bf16.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.bfloat16, device_map=device)
messages = [{"role": "user", "content": "Explain what a TPU is in two sentences."}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                       return_dict=True, return_tensors="pt").to(device)
for run in range(2):
    t0 = time.perf_counter()
    out = llm.generate(**inputs, max_new_tokens=80, do_sample=False)
    sync()
    n_new = out.shape[1] - inputs.input_ids.shape[1]
    print(f"run {run}: {n_new} tokens in {time.perf_counter()-t0:.1f}s")
print(tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))

Generation works, but it is not where a TPU shines: each new token is a small computation with a new sequence length, which suits a GPU's ready-made kernels better than a compiler. (Static KV caches plus `torch.compile`, the usual trick to speed this up on GPUs, did not work with TorchTPU 0.2 when this notebook was written.)

### Fine-tuning with LoRA

PEFT's LoRA works on the TPU too. In our tests the 🤗 `Trainer` did **not** (it left the batches on the CPU: `embedding(): expected tensor to be on tpu, got cpu`), and TRL's `SFTTrainer` is built on top of it. A plain training loop like the ones in 02 and 03 works fine, so that is what we use: teach Qwen3-0.6B to answer additions in a fixed format.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)   # not preinstalled on the TPU runtime
from peft import LoraConfig, get_peft_model

lora_model = get_peft_model(llm, LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
lora_model.print_trainable_parameters()

def example(a, b):
    messages = [{"role": "user", "content": f"What is {a} + {b}?"}, {"role": "assistant", "content": f"{a} + {b} = {a + b}"}]
    return tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)

texts = [example(a, b) for a, b in torch.randint(0, 1000, (2048, 2)).tolist()]
enc = tokenizer(texts, padding="max_length", max_length=48, truncation=True, return_tensors="pt")  # fixed length!
labels = enc.input_ids.masked_fill(enc.attention_mask == 0, -100)          # no loss on padding
optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=2e-4)
lora_model.train()
t0 = time.perf_counter()
for step, i in enumerate(range(0, len(texts), 32)):
    batch = {"input_ids": enc.input_ids[i:i + 32].to(device), "attention_mask": enc.attention_mask[i:i + 32].to(device),
             "labels": labels[i:i + 32].to(device)}
    loss = lora_model(**batch).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if step % 16 == 0:
        print(f"step {step:3d}  loss {loss.item():.3f}  {time.perf_counter()-t0:.0f}s")

In [ ]:
lora_model.eval()
q = tokenizer.apply_chat_template([{"role": "user", "content": "What is 123 + 456?"}], add_generation_prompt=True,
                                  enable_thinking=False, return_dict=True, return_tensors="pt").to(device)
out = lora_model.generate(**q, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(out[0, q.input_ids.shape[1]:], skip_special_tokens=True))

## PyTorch and 🤗 Transformers on a TPU: pros and cons

What we tested on a Colab TPU v5e-1 (torch 2.14, TorchTPU 0.2, transformers 5.22, November 2026):

| | Works? |
|---|---|
| Plain PyTorch: tensors, autograd, optimizers, `nn` layers, SDPA attention | ✅ |
| `torch.compile(backend="tpu")`, bf16 autocast | ✅ |
| `AdaptiveMaxPool2d` backward | ❌ not implemented (use `MaxPool2d`) |
| BatchNorm running statistics | ❌ not updated (recalibrate, see above) |
| 🤗 `from_pretrained(..., device_map="tpu")`, `generate` | ✅ (eager) |
| static KV cache + `torch.compile` for generation | ❌ |
| PEFT LoRA with your own training loop | ✅ |
| 🤗 `Trainer` (and TRL, built on it) | ❌ batches stay on the CPU |
| `bitsandbytes` 4-bit, FlashAttention, Triton kernels, FlexAttention | ❌ CUDA/Triton only |

**Pros**

* **Speed for the right workloads**: large dense computations in bf16 with fixed shapes. When this notebook was written, a TPU v5e-1 vs. the T4 in this course: matrix multiplication about 5x faster (~160 vs. ~30 TFLOPS), the ResNet9 epoch about 2x faster (4.8 s vs. 9 s, same accuracy), and the GPT training about 2.5x faster (74 s vs. ~185 s for 1500 steps).
* **Same PyTorch code**: with `torch.accelerator`, most code only needs `device = "tpu"`. Nothing to install on Colab.
* **bf16 without `GradScaler`**, the same 16 GB of memory as a T4, and a host with far more CPU cores and RAM for data loading.
* **Availability**: when no T4 is free, a TPU often is; and the same code scales to multi-chip TPU pods (data and tensor parallelism) on Google Cloud.

**Cons**

* **A young backend**: missing operators, and silent bugs such as the BatchNorm statistics. You must check results, not just that the code runs.
* **Compilation**: the first steps are slow, and changing shapes causes recompilation. Pad to fixed lengths, use `drop_last`, avoid shape-dependent Python logic.
* **The 🤗 high-level tools lag behind**: `Trainer`/TRL did not work in our tests, so you write your own training loop; fast generation tricks for GPUs do not carry over, and token-by-token generation is slower than on a T4.
* **No CUDA ecosystem**: quantization with `bitsandbytes`, FlashAttention, custom Triton or CUDA kernels, and anything built on them, are unavailable.
* **Fewer examples and answers online** than for CUDA, so debugging takes longer.

Rule of thumb: on Colab, **use the T4 when you can get one** (everything in this course was written for it); **use the TPU** when the T4 is not available or for heavy training of standard models with fixed shapes, written as a plain PyTorch loop.

## What to watch out for on a TPU

* **Use bf16, not fp16.** bf16 has the same range as fp32, so no `GradScaler` is needed.
* **Keep tensor shapes fixed.** Every new shape is compiled once; changing shapes (a smaller last batch, variable sequence lengths) cause recompilation. Use `drop_last=True`, pad sequences to fixed lengths (or a few bucket sizes).
* **Use `torch.compile(model, backend="tpu")` for training loops** with fixed shapes; keep eager mode for code whose shapes keep changing, such as generating text one token at a time.
* **Avoid needless synchronization.** `loss.item()`, `print(tensor)` or converting to NumPy make the CPU wait for the TPU; do it every N steps, not every step.
* **Some operators are missing or buggy** in a young backend (`AdaptiveMaxPool2d` backward, BatchNorm statistics above). Missing operators raise an error naming them; bugs may only show up as wrong results, so always check accuracy, not just that the code runs.
* **GPU-only libraries do not work**: anything built on CUDA or Triton kernels, such as `bitsandbytes` (4-bit quantization in 04-LLM-Apps), FlashAttention, or custom CUDA extensions.
* **A TPU v5e chip has 16 GB of memory**, about the same as a T4.
* Data loading: the TPU host has many CPU cores, so a `DataLoader` with more workers (03) is much less of a bottleneck than on the T4 runtime.

## Further reading

* [TorchTPU on GitHub](https://github.com/google-pytorch/torch_tpu): the native PyTorch backend used here
* [Hugging Face ❤️ TorchTPU](https://huggingface.co/blog/hf-torchtpu): Transformers and Diffusers on TPUs
* [`torch.accelerator`](https://docs.pytorch.org/docs/stable/accelerator.html): device-agnostic PyTorch code
* [Colab: TPU v5e-1 in the free tier](https://github.com/googlecolab/colabtools/issues/5566)
* [PyTorch/XLA](https://docs.pytorch.org/xla/): the older way to run PyTorch on TPUs, which you will still see in many tutorials